In [ ]:
class Trie:
    def __init__(self):
        self.isEnd = False
        self.children = dict()

    def add_word(self, word):
        trie = self
        for char in word:
            if char not in trie.children:
                trie.children[char] = Trie()
            trie = trie.children[char]
        
        trie.isEnd = True

    def search(self, word):
        trie = self
        for char in word:
            if char not in trie.children:
                return False
            trie = trie.children[char]
        
        return trie.isEnd
    
    def autocomplete(self, prefix):
        trie = self
        for char in prefix:
            if char not in trie.children:
                return []
            trie = trie.children[char]

        result = []
        def autocomplete_util(trie, suffix):
            if trie.isEnd:
                result.append(suffix)

            for char in trie.children:
                autocomplete_util(trie.children[char], suffix + char)
        
        autocomplete_util(trie, prefix)
        return result
    
    def wildcard_search(self, word):
        trie = self

        def wc_search_util(trie, j):
            for i in range(j, len(word)):
                char = word[i]
                if char != '.':
                    if char not in trie.children:
                        return False
                    trie = trie.children[char]
                else:
                    for child in trie.children.values():
                        if wc_search_util(child, i + 1):
                            return True
                    return False
            
            return trie.isEnd
        
        return wc_search_util(trie, 0)

    # O(k.n^2); k = total stored words, n = len of word
    # path + c/path += c is O(n) operation as py strs are immutable
    def wildcard_search2(self, word):
        trie = self
        result = []

        def wc_search_util(trie, j, path):
            for i in range(j, len(word)):
                char = word[i]
                if char != '.':
                    if char not in trie.children:
                        return
                    path += char
                    trie = trie.children[char]
                else:
                    for c, child in trie.children.items():
                        wc_search_util(child, i + 1, path + c)
                    return
            
            if trie.isEnd:
                result.append(path)
        
        wc_search_util(trie, 0, '')
        return result

    # TC: O(k.n)
    def wildcard_search2_opt(self, word):
        trie = self
        res = []

        def util(trie, i, path: list[str]):
            for j in range(i, len(word)):
                char = word[j]
                if char != '.':
                    if char not in trie.children:
                        return
                    path.append(char)
                    trie = trie.children[char]
                else:
                    for c, children in trie.children.items():
                        original_len = len(path)
                        path.append(c)
                        util(children, j + 1, path)
                        del path[original_len:]

            if trie.isEnd:
                res.append(''.join(path))

        util(trie, 0, [])
        return res

In [14]:
trie = Trie()

trie.add_word('bun')
trie.add_word('bundle')
trie.add_word('bungle')
trie.add_word('burn')

trie.wildcard_search2_opt('bun.le')
# trie.autocomplete('bu')

['bundle', 'bungle']

### Word Break II

In [ ]:
class Trie:
    def __init__(self):
        self.children = {}
        self.is_word = False

    def add_word(self, word):
        trie = self
        for char in word:
            if char not in trie.children:
                trie.children[char] = Trie()
            trie = trie.children[char]

        trie.is_word = True

def find_words(board: list[list[str]], words: list[str]):
    trie = Trie()
    for word in words:
        trie.add_word(word)

    ROWS, COLS = len(board), len(board[0])
    res = set()
    visit = set()
    def explore(r, c, node, prefix):
        if (not (0 <= r < ROWS) or 
            not (0 <= c < COLS) or 
            board[r][c] not in node.children or 
            (r, c) in visit):
            return 

        visit.add((r, c))

        char = board[r][c]
        node = node.children[char]
        new_prefix = prefix + char
        if node.is_word:
            res.add(new_prefix)

        iters = [(1, 0), (-1, 0), (0, 1), (0, -1)]
        for dr, dc in iters:
            nr, nc = r + dr, c + dc
            explore(nr, nc, node, new_prefix)

        visit.remove((r, c))

    for r in range(ROWS):
        for c in range(COLS):
            explore(r, c, trie, '')

    return res

board = [
  ["a","b","c","d"],
  ["s","a","a","t"],
  ["a","c","k","e"],
  ["a","c","d","n"]
]
words = ["bat","cat","back","backend","stack"]
find_words(board, words)

['back', 'backend', 'cat']